# Лабораторная работа 6
## Жадная стратегия и динамическое программирование

Выполнение заданий по порядку.


## Задача №1: Ручное заполнение таблицы для задачи о рюкзаке

**Вариант 2:**
- вода: 1 фунт, ценность 10
- книга: 2 фунта, ценность 3
- еда: 2 фунта, ценность 9
- куртка: 1 фунт, ценность 5
- камера: 4 фунта, ценность 6

**Емкость рюкзака:** 6 фунтов

Таблица динамического программирования:


In [ ]:
import pandas as pd

# Предметы: (вес, ценность, название) - Вариант 2
items = [
    (1, 10, "вода"),
    (2, 3, "книга"),
    (2, 9, "еда"),
    (1, 5, "куртка"),
    (4, 6, "камера")
]

capacity = 6
n = len(items)

# Создаем таблицу ДП: dp[i][w] = максимальная ценность при использовании первых i предметов и весе w
dp = [[0] * (capacity + 1) for _ in range(n + 1)]

# Заполняем таблицу
for i in range(1, n + 1):
    weight, value, name = items[i - 1]
    for w in range(capacity + 1):
        # Не берем предмет i
        dp[i][w] = dp[i - 1][w]
        # Берем предмет i, если он помещается
        if w >= weight:
            dp[i][w] = max(dp[i][w], dp[i - 1][w - weight] + value)

# Выводим таблицу
columns = [f"Вес {w}" for w in range(capacity + 1)]
rows = ["Нет предметов"] + [f"{items[i][2]}" for i in range(n)]
df = pd.DataFrame(dp, columns=columns, index=rows)
print("Таблица динамического программирования:")
print(df)
print(f"\nОптимальная ценность: {dp[n][capacity]}")

# Восстанавливаем набор предметов
selected = []
w = capacity
for i in range(n, 0, -1):
    if dp[i][w] != dp[i - 1][w]:
        selected.append(items[i - 1][2])
        w -= items[i - 1][0]

print(f"Выбранные предметы: {selected}")
print(f"Общий вес: {sum(items[i][0] for i in range(n) if items[i][2] in selected)}")
print(f"Общая ценность: {dp[n][capacity]}")


## Задача №2: Программа для задачи о рюкзаке (3 алгоритма)


In [ ]:
import itertools
import time
import matplotlib.pyplot as plt
import numpy as np

def knapsack_bruteforce(capacity, items):
    """Полный перебор: O(2^n)"""
    n = len(items)
    max_value = 0
    best_combination = []
    
    for r in range(1, n + 1):
        for combo in itertools.combinations(range(n), r):
            total_weight = sum(items[i][0] for i in combo)
            if total_weight <= capacity:
                total_value = sum(items[i][1] for i in combo)
                if total_value > max_value:
                    max_value = total_value
                    best_combination = combo
    
    return max_value, [items[i] for i in best_combination]

def knapsack_greedy(capacity, items):
    """Жадный алгоритм: O(n log n) - сортировка по ценности/весу"""
    # Сортируем по убыванию ценности на единицу веса
    sorted_items = sorted(items, key=lambda x: x[1]/x[0], reverse=True)
    selected = []
    total_weight = 0
    total_value = 0
    
    for item in sorted_items:
        if total_weight + item[0] <= capacity:
            selected.append(item)
            total_weight += item[0]
            total_value += item[1]
    
    return total_value, selected

def knapsack_dp(capacity, items):
    """Динамическое программирование: O(n * capacity)"""
    n = len(items)
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    
    for i in range(1, n + 1):
        weight, value = items[i - 1][0], items[i - 1][1]
        for w in range(capacity + 1):
            dp[i][w] = dp[i - 1][w]
            if w >= weight:
                dp[i][w] = max(dp[i][w], dp[i - 1][w - weight] + value)
    
    # Восстанавливаем набор
    selected = []
    w = capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            selected.append(items[i - 1])
            w -= items[i - 1][0]
    
    return dp[n][capacity], selected

# Тестирование (Вариант 2)
capacity = 6
items = [(1, 10, "вода"), (2, 3, "книга"), (2, 9, "еда"), (1, 5, "куртка"), (4, 6, "камера")]

print("=== Тестирование алгоритмов ===")
print(f"Емкость рюкзака: {capacity}")
print(f"Предметы: {items}\n")

val_bf, items_bf = knapsack_bruteforce(capacity, items)
print(f"Полный перебор: ценность = {val_bf}, предметы = {items_bf}")

val_gr, items_gr = knapsack_greedy(capacity, items)
print(f"Жадный алгоритм: ценность = {val_gr}, предметы = {items_gr}")

val_dp, items_dp = knapsack_dp(capacity, items)
print(f"ДП: ценность = {val_dp}, предметы = {items_dp}")

print("\n=== Вычислительная сложность ===")
print("Полный перебор: O(2^n)")
print("Жадный алгоритм: O(n log n)")
print("Динамическое программирование: O(n * capacity)")


In [ ]:
# Графики времени выполнения
def measure_time(func, capacity, items):
    start = time.time()
    func(capacity, items)
    return time.time() - start

sizes = [5, 10, 15, 20]
times_bf = []
times_gr = []
times_dp = []

for n in sizes:
    items = [(np.random.randint(1, 10), np.random.randint(1, 20), f"item{i}") for i in range(n)]
    cap = 20
    
    times_bf.append(measure_time(knapsack_bruteforce, cap, items))
    times_gr.append(measure_time(knapsack_greedy, cap, items))
    times_dp.append(measure_time(knapsack_dp, cap, items))

plt.figure(figsize=(10, 6))
plt.plot(sizes, times_bf, 'o-', label='Полный перебор O(2^n)')
plt.plot(sizes, times_gr, 's-', label='Жадный O(n log n)')
plt.plot(sizes, times_dp, '^-', label='ДП O(n*capacity)')
plt.xlabel('Количество предметов')
plt.ylabel('Время выполнения (сек)')
plt.title('Сравнение времени выполнения алгоритмов')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Примеры: когда жадный алгоритм оптимален и когда нет
print("=== Пример 1: Жадный алгоритм дает оптимальное решение ===")
items1 = [(2, 10, "A"), (3, 15, "B"), (1, 5, "C")]
cap1 = 4
val_gr1, _ = knapsack_greedy(cap1, items1)
val_dp1, _ = knapsack_dp(cap1, items1)
print(f"Предметы: {items1}, емкость: {cap1}")
print(f"Жадный: {val_gr1}, ДП: {val_dp1}, Оптимально: {val_gr1 == val_dp1}")

print("\n=== Пример 2: Жадный алгоритм дает неоптимальное решение ===")
items2 = [(10, 60, "A"), (20, 100, "B"), (30, 120, "C")]
cap2 = 50
val_gr2, items_gr2 = knapsack_greedy(cap2, items2)
val_dp2, items_dp2 = knapsack_dp(cap2, items2)
print(f"Предметы: {items2}, емкость: {cap2}")
print(f"Жадный: {val_gr2} ({items_gr2})")
print(f"ДП: {val_dp2} ({items_dp2})")
print(f"Оптимально: {val_gr2 == val_dp2}")


## Задача №3: Ручное заполнение таблицы для самой длинной общей подстроки

Строки: **blue** и **clue**


In [ ]:
def longest_common_substring_table(s1, s2):
    """Создает таблицу для самой длинной общей подстроки"""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    max_len = 0
    end_pos = 0
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                if dp[i][j] > max_len:
                    max_len = dp[i][j]
                    end_pos = i
            else:
                dp[i][j] = 0
    
    # Выводим таблицу
    print("Таблица для самой длинной общей подстроки:")
    print("   ", " ".join([f"  {c}" for c in s2]))
    for i in range(m + 1):
        row = [f"{s1[i-1]} " if i > 0 else "  "]
        for j in range(n + 1):
            row.append(f"{dp[i][j]:2d}")
        print(" ".join(row))
    
    result = s1[end_pos - max_len:end_pos] if max_len > 0 else ""
    print(f"\nСамая длинная общая подстрока: '{result}' (длина: {max_len})")
    return result, max_len

s1, s2 = "blue", "clue"
longest_common_substring_table(s1, s2)


## Задача №4: Программа для самой длинной общей подстроки


In [ ]:
def find_most_similar_substring(word, word_list):
    """Находит самое похожее слово по длине общей подстроки"""
    max_len = 0
    best_word = None
    
    for candidate in word_list:
        m, n = len(word), len(candidate)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        current_max = 0
        
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if word[i-1] == candidate[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                    current_max = max(current_max, dp[i][j])
                else:
                    dp[i][j] = 0
        
        if current_max > max_len:
            max_len = current_max
            best_word = candidate
    
    return best_word, max_len

# Пример использования
word_with_error = "blu"
similar_words = ["blue", "clue", "glue", "blur"]

best_match, length = find_most_similar_substring(word_with_error, similar_words)
print(f"Слово с ошибкой: '{word_with_error}'")
print(f"Список похожих слов: {similar_words}")
print(f"Самое похожее слово: '{best_match}' (длина общей подстроки: {length})")


In [ ]:
# Интерактивный ввод
print("=== Интерактивный режим ===")
word_with_error = input("Введите слово с ошибкой: ")
similar_words = input("Введите список похожих слов через запятую: ").split(",")
similar_words = [w.strip() for w in similar_words]

best_match, length = find_most_similar_substring(word_with_error, similar_words)
print(f"\nСамое похожее слово: '{best_match}' (длина общей подстроки: {length})")


## Задача №5: Программа для самой длинной общей подпоследовательности


In [ ]:
def longest_common_subsequence(s1, s2):
    """Находит длину самой длинной общей подпоследовательности"""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    return dp[m][n]

def find_most_similar_lcs(word, word_list):
    """Находит самое похожее слово по длине общей подпоследовательности"""
    max_len = 0
    best_word = None
    
    for candidate in word_list:
        lcs_len = longest_common_subsequence(word, candidate)
        if lcs_len > max_len:
            max_len = lcs_len
            best_word = candidate
    
    return best_word, max_len

# Пример использования
print("=== Интерактивный режим ===")
word_with_error = input("Введите слово с ошибкой: ")
similar_words = input("Введите список похожих слов через запятую: ").split(",")
similar_words = [w.strip() for w in similar_words]

best_match, length = find_most_similar_lcs(word_with_error, similar_words)
print(f"\nСамое похожее слово: '{best_match}' (длина общей подпоследовательности: {length})")


## Задача №6: Программа для упаковки в ящики (4 жадные стратегии)


In [ ]:
def first_fit(items):
    """Первый подходящий ящик: O(n*m)"""
    boxes = []
    for item in items:
        placed = False
        for box in boxes:
            if sum(box) + item <= 1.0:
                box.append(item)
                placed = True
                break
        if not placed:
            boxes.append([item])
    return boxes

def best_fit(items):
    """Наиболее подходящий ящик: O(n*m)"""
    boxes = []
    for item in items:
        best_idx = -1
        min_space = 1.0
        for i, box in enumerate(boxes):
            space = 1.0 - sum(box)
            if space >= item and space < min_space:
                min_space = space
                best_idx = i
        if best_idx != -1:
            boxes[best_idx].append(item)
        else:
            boxes.append([item])
    return boxes

def next_fit(items):
    """Следующий подходящий ящик: O(n)"""
    boxes = []
    if not items:
        return boxes
    current_box = [items[0]]
    for item in items[1:]:
        if sum(current_box) + item <= 1.0:
            current_box.append(item)
        else:
            boxes.append(current_box)
            current_box = [item]
    boxes.append(current_box)
    return boxes

def worst_fit(items):
    """Наименее подходящий ящик: O(n*m)"""
    boxes = []
    for item in items:
        worst_idx = -1
        max_space = -1
        for i, box in enumerate(boxes):
            space = 1.0 - sum(box)
            if space >= item and space > max_space:
                max_space = space
                worst_idx = i
        if worst_idx != -1:
            boxes[worst_idx].append(item)
        else:
            boxes.append([item])
    return boxes

# Тестирование на примере из задания
test_items = [0.5, 0.7, 0.3, 0.9, 0.6, 0.8, 0.1, 0.4, 0.2, 0.5]

print("Тестовый набор:", test_items)
print(f"\nПервый подходящий: {len(first_fit(test_items))} ящиков")
print(f"Наиболее подходящий: {len(best_fit(test_items))} ящиков")
print(f"Следующий подходящий: {len(next_fit(test_items))} ящиков")
print(f"Наименее подходящий: {len(worst_fit(test_items))} ящиков")


In [ ]:
# Генерация случайных наборов и сравнение стратегий
np.random.seed(42)
sizes = [50, 100, 200, 500]
results = []

for size in sizes:
    items = np.random.uniform(0.1, 0.9, size).round(1).tolist()
    
    ff_count = len(first_fit(items))
    bf_count = len(best_fit(items))
    nf_count = len(next_fit(items))
    wf_count = len(worst_fit(items))
    
    results.append({
        'Размер': size,
        'Первый подходящий': ff_count,
        'Наиболее подходящий': bf_count,
        'Следующий подходящий': nf_count,
        'Наименее подходящий': wf_count
    })

df_results = pd.DataFrame(results)
print("Результаты сравнения стратегий:")
print(df_results.to_string(index=False))


## Задача №7: Задача о размене денег (жадный алгоритм и ДП)


In [ ]:
def greedy_change(n, denominations):
    """Жадный алгоритм: O(n)"""
    denominations = sorted(denominations, reverse=True)
    result = {}
    for d in denominations:
        count = n // d
        if count > 0:
            result[d] = count
            n %= d
    return result if n == 0 else None

def dp_change(n, denominations):
    """Динамическое программирование: O(n * len(denominations))"""
    denominations = sorted(denominations)
    dp = [float('inf')] * (n + 1)
    dp[0] = 0
    parent = [-1] * (n + 1)
    
    for i in range(1, n + 1):
        for d in denominations:
            if i >= d and dp[i - d] + 1 < dp[i]:
                dp[i] = dp[i - d] + 1
                parent[i] = d
    
    if dp[n] == float('inf'):
        return None
    
    # Восстанавливаем набор
    result = {}
    i = n
    while i > 0:
        d = parent[i]
        result[d] = result.get(d, 0) + 1
        i -= d
    
    return result

denominations = [1, 3, 4, 10, 50, 100]

print("Номиналы:", denominations)

# Пример 1: обычный случай
n = 87
gr_result = greedy_change(n, denominations)
dp_result = dp_change(n, denominations)
print(f"\nПример 1 - Сумма: {n}")
print(f"Жадный алгоритм: {gr_result}, количество купюр: {sum(gr_result.values()) if gr_result else 'невозможно'}")
print(f"ДП: {dp_result}, количество купюр: {sum(dp_result.values()) if dp_result else 'невозможно'}")

# Пример, когда жадный алгоритм неоптимален
print("\n=== Пример неоптимальности жадного алгоритма ===")
test_sum = 6
gr_test = greedy_change(test_sum, denominations)
dp_test = dp_change(test_sum, denominations)
print(f"Сумма: {test_sum}")
print(f"Жадный: {gr_test} ({sum(gr_test.values()) if gr_test else 'невозможно'} купюр)")
print(f"ДП: {dp_test} ({sum(dp_test.values()) if dp_test else 'невозможно'} купюр)")
print(f"Оптимально: {sum(gr_test.values()) == sum(dp_test.values()) if gr_test and dp_test else False}")
print("\nПримечание: Для суммы 6 жадный алгоритм выберет 4+1+1 (3 купюры),")
print("а оптимальное решение - 3+3 (2 купюры)")


## Задача №8: Банкомат с ограниченным количеством купюр


In [ ]:
def atm_change(n, denominations, available):
    """Размен с учетом ограниченного количества купюр (жадный алгоритм)"""
    denominations = sorted(denominations, reverse=True)
    result = {}
    remaining = available.copy()
    
    for d in denominations:
        count = min(n // d, remaining.get(d, 0))
        if count > 0:
            result[d] = count
            n -= count * d
            remaining[d] -= count
    
    return (result, remaining) if n == 0 else (None, remaining)

def simulate_atm(clients, initial_available):
    """Симуляция работы банкомата"""
    denominations = [1, 3, 4, 10, 50, 100]
    available = initial_available.copy()
    
    print(f"\n=== Начальное состояние банкомата ===")
    print(f"Доступные купюры: {available}")
    
    for i, amount in enumerate(clients):
        print(f"\n--- Клиент {i+1}, сумма: {amount} ---")
        result, remaining = atm_change(amount, denominations, available)
        
        if result:
            print(f"Выдано: {result}")
            print(f"Количество купюр: {sum(result.values())}")
            available = remaining
            print(f"Остаток в банкомате: {available}")
        else:
            print("Невозможно выдать сумму - нехватка купюр")
    
    print(f"\n=== Финальное состояние банкомата ===")
    print(f"Остаток: {available}")

# Пример использования (раскомментируйте для интерактивного режима)
# num_clients = int(input("Введите количество клиентов: "))
# clients = [int(input(f"Клиент {i+1}, сумма: ")) for i in range(num_clients)]
# available = {d: int(input(f"Номинал {d}: ")) for d in sorted([1,3,4,10,50,100], reverse=True)}
# simulate_atm(clients, available)


In [ ]:
# Пример запуска с предустановленными данными
print("=== Пример 1: Успешные операции ===")
denominations = [1, 3, 4, 10, 50, 100]
available1 = {100: 5, 50: 10, 10: 20, 4: 30, 3: 40, 1: 100}
clients1 = [150, 87, 234]

simulate_atm(clients1, available1)

print("\n=== Пример 2: Нехватка купюр ===")
available2 = {100: 1, 50: 1, 10: 2, 4: 1, 3: 1, 1: 5}
clients2 = [200, 50]

simulate_atm(clients2, available2)
